# r.noaa.atlas14

This notebook runs the examples from the *r.noaa.atlas14* manual page and visualizes the output. The tool downloads and imports [NOAA Atlas 14](https://hdsc.nws.noaa.gov/pfds/) precipitation-frequency data and supports two acquisition modes:

- **mode=point** queries the NOAA Precipitation Frequency Data Server (PFDS) for one or more longitude/latitude pairs and writes the estimates as JSON or CSV, optionally as a GRASS vector point map.
- **mode=grid** discovers, downloads, and imports NOAA Atlas 14 GIS-compatible grid archives, rescaling them to the requested statistic/units.

Capabilities demonstrated below:

| Capability | Where |
|---|---|
| Point query to JSON + vector map | Point mode |
| Depth-duration-frequency curves | Point mode |
| Intensity, CSV output, custom separator | Point mode |
| Confidence bounds (`bound=all`) | Point mode |
| Multiple points in one call | Point mode |
| Region-center fallback (no coordinates) | Point mode |
| List archives without downloading (`-l`) | Grid mode |
| Import + rescaled units, title, and history | Grid mode |
| Intensity rescaling | Grid mode |
| Filtered batch import + manifest | Grid mode |

## Setup

We use the NC SPM sample project and run in a temporary mapset.

In [ ]:
import json
import os
import subprocess
import sys

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import pandas as pd
import matplotlib.pyplot as plt
import grass.script as gs
import grass.jupyter as gj

gj.init("~/grassdata/nc_spm_full_v2alpha2/PERMANENT")

# Create and switch to a throwaway mapset for this notebook run.
tmp_mapset = f"tmp_noaa_atlas14_{os.getpid()}"
gs.run_command("g.mapset", flags="c", mapset=tmp_mapset)
gs.run_command("g.region", raster="elevation")

## Point mode: PFDS query for Raleigh, NC

Fetch the expected precipitation depth for downtown Raleigh and write both JSON and a GRASS vector point map that carries the full tables as JSON attributes.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="point",
    coordinates="-78.6382,35.7796",
    statistic="depth",
    units="english",
    series="pds",
    bound="expected",
    format="json",
    output="/tmp/raleigh_atlas14.json",
    vector_output="atlas14_raleigh",
    overwrite=True,
)

In [ ]:
# Inspect the JSON payload.
with open("/tmp/raleigh_atlas14.json") as fh:
    data = json.load(fh)

print("Return periods (years):", data["table"]["return_periods_years"])
print("First few rows:")
for row in data["table"]["rows"][:5]:
    print(f"  {row['duration']:>8}  {row['values']}")

### Depth-duration-frequency curves

Plot the expected precipitation depth against duration, one curve per return period, straight from the JSON table.

In [ ]:
table = data["table"]
return_periods = table["return_periods_years"]
durations = [row["duration"] for row in table["rows"]]

fig, ax = plt.subplots(figsize=(8, 5))
for rp in return_periods:
    depths = [row["values"].get(str(rp)) for row in table["rows"]]
    ax.plot(range(len(durations)), depths, marker="o", label=f"{rp}-yr")

ax.set_xticks(range(len(durations)))
ax.set_xticklabels(durations, rotation=45, ha="right")
ax.set_xlabel("Duration")
ax.set_ylabel("Precipitation depth (inches)")
ax.set_title("NOAA Atlas 14 depth-duration-frequency, Raleigh NC (expected)")
ax.legend(title="Return period", ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig

### Intensity, CSV output, and a custom separator

Switch `statistic=intensity`, write tab-separated CSV, and read it back with pandas.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="point",
    coordinates="-78.6382,35.7796",
    statistic="intensity",
    units="english",
    series="pds",
    bound="expected",
    format="csv",
    separator="tab",
    output="/tmp/raleigh_intensity.csv",
    overwrite=True,
)

idf = pd.read_csv("/tmp/raleigh_intensity.csv", sep="\t")
idf.head()

### Confidence bounds

`bound=all` returns the expected value plus the upper and lower bounds of the 90% confidence interval. Here we compare the 100-year, 24-hour estimate across the three bounds.

In [ ]:
bounds_json = gs.read_command(
    "r.noaa.atlas14",
    mode="point",
    coordinates="-78.6382,35.7796",
    statistic="depth",
    bound="all",
    format="json",
    flags="c",
)
bounds = json.loads(bounds_json)
for section in ("expected", "upper", "lower"):
    rows = bounds["tables"][section]["rows"]
    row24 = next(r for r in rows if "24-hr" in r["duration"])
    print(f"{section:>8}: 100-yr 24-hr = {row24['values']['100']} in")

### Multiple points in one call

Pass several `lon,lat` pairs (Raleigh, Charlotte, Wilmington). The CSV gains `lon,lat,bound` columns, and the vector map gets one feature per point with the per-point JSON tables as attributes.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="point",
    coordinates="-78.6382,35.7796,-80.8431,35.2271,-77.8868,34.2257",
    statistic="depth",
    bound="expected",
    format="csv",
    output="/tmp/nc_cities_idf.csv",
    vector_output="atlas14_nc_cities",
    overwrite=True,
)

cities = pd.read_csv("/tmp/nc_cities_idf.csv")
n_points = cities[["lon", "lat"]].drop_duplicates().shape[0]
print("CSV rows:", len(cities), "| distinct points:", n_points)
cities.head()

In [ ]:
# The vector map carries one feature per queried point.
print(
    gs.read_command(
        "v.db.select", map="atlas14_nc_cities", columns="cat,lon,lat", format="csv"
    )
)

### Region-center fallback

With no `coordinates=`, the tool queries the center of the current computational region, reprojecting the region bounds to WGS84 lon/lat so it works from any project CRS.

In [ ]:
gs.run_command("g.region", raster="elevation")
center_json = gs.read_command(
    "r.noaa.atlas14", mode="point", format="json", bound="expected", flags="c"
)
center = json.loads(center_json)
print("Queried lon/lat:", center["request"]["lon"], center["request"]["lat"])
print("Request URL:", center["request"]["url"])

## Grid mode

North Carolina falls in NOAA Atlas 14 Volume 2 (Ohio River Basin and surrounding states), region code `orb`, so we use that volume below.

### List matching archives without downloading

The `-l` flag reports which archives match the filters (as JSON lines) so you can scope a download before committing to it.

In [ ]:
listing = gs.read_command(
    "r.noaa.atlas14",
    mode="grid",
    region="orb",
    durations="24hr",
    aris="2,10,100",
    bound="expected",
    flags="l",
)
matches = [json.loads(line) for line in listing.splitlines() if line.strip()]
print("matching archives:", len(matches))
for m in matches:
    print(
        f"  {m['filename']:>18}  ari={m['ari']}  dur={m['duration']}  bound={m['bound']}"
    )

### Import a single archive

Download one archive directly by URL and import the rescaled raster (depth in inches) with `-i` (uses *r.import* so it is reprojected into the NC SPM project). The full grid extent is imported regardless of the current region.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="grid",
    archive_url="https://hdsc.nws.noaa.gov/pub/hdsc/data/orb/orb100yr24ha.zip",
    output_prefix="a14",
    flags="i",
    overwrite=True,
)

In [ ]:
raster_name = gs.list_grouped(type="raster", pattern="a14_*")[gs.gisenv()["MAPSET"]][0]
print("Imported:", raster_name)

# r.info shows the rescaled units and the descriptive title set by the tool.
info = gs.raster_info(raster_name)
print("units:", info["units"], "| title:", info["title"])
print("min/max (inches):", round(info["min"], 3), round(info["max"], 3))

In [ ]:
# Processing history is recorded on the output raster.
print(gs.read_command("r.info", map=raster_name, flags="h"))

In [ ]:
with gs.MaskManager(mask_name="ncmask_500m"):
    gs.run_command("g.region", raster=raster_name)
    gs.run_command("r.colors", map=raster_name, color="ryb", flags="e")
    grid_map = gj.Map(use_region=False)
    grid_map.d_rast(map=raster_name)
    grid_map.d_vect(
        map="atlas14_nc_cities", fill_color="white", size=10, icon="basic/circle"
    )
    grid_map.d_legend(raster=raster_name, at=(5, 50, 5, 8), flags="b")
    grid_map.d_barscale()
    grid_map.show()

### Intensity rescaling

Re-import the same archive with `statistic=intensity`. NOAA only publishes depth, so the tool divides by the duration to produce in/hr automatically.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="grid",
    archive_url="https://hdsc.nws.noaa.gov/pub/hdsc/data/orb/orb100yr24ha.zip",
    statistic="intensity",
    units="english",
    output_prefix="a14int",
    flags="i",
    overwrite=True,
)
intensity_rast = gs.list_grouped(type="raster", pattern="a14int_*")[
    gs.gisenv()["MAPSET"]
][0]
iinfo = gs.raster_info(intensity_rast)
print(intensity_rast)
print(
    "units:",
    iinfo["units"],
    "| min/max (in/hr):",
    round(iinfo["min"], 5),
    round(iinfo["max"], 5),
)

### Filtered batch import with a manifest

Autodiscover the `orb` volume, filter to a duration and a couple of recurrence intervals, and write a manifest CSV describing every imported raster.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="grid",
    region="orb",
    durations="12hr",
    aris="10,100",
    bound="expected",
    output_prefix="batch",
    output="/tmp/a14_manifest.csv",
    flags="i",
    overwrite=True,
)
manifest = pd.read_csv("/tmp/a14_manifest.csv")
manifest[["map", "duration", "ari", "statistic", "units", "region"]]

### Clean up

Remove the temporary mapset created for this run.

In [ ]:
gs.run_command("g.mapset", mapset="PERMANENT")
gs.run_command("g.mapsets", operation="remove", mapset=tmp_mapset)
# To fully delete it, remove its directory from the project on disk.